In [0]:
from pyspark.sql.functions import regexp_extract, regexp_replace, trim, col, split, concat
from pyspark.sql import functions as F

# Load all CSVs from the kenpom directory
kenpom_path = "/Volumes/workspace/marchmadnesslandingzone/kenpom/*.csv"
df = spark.read.option("header", True).csv(kenpom_path)

display(df)


In [0]:
# Extract the seed (last number following space at end of Team string)
df = df.withColumn("Seed", regexp_extract(col("Team"), r" (\d+)$", 1))

# Only keep rows with extracted Seed
kenpom_seeded = df.filter(col("Seed") != "")

# Clean Team name: remove trailing numbers, spaces, and periods
kenpom_seeded = kenpom_seeded.withColumn(
    "Team",
    trim(regexp_replace(regexp_replace(col("Team"), r"(\d+)$", ""), r"[.]", ""))
)

# Split Record column to get Wins
kenpom_seeded = kenpom_seeded.withColumn(
    "Wins", split(col("Record"), "-").getItem(0)
)

# Select the major columns needed
target_cols = [
     "Team", "Year", "Seed", "Wins",
    "AdjEM", "AdjO", "AdjD", "SoS_AdjEM", "OppO", "OppD"
]
df_target = kenpom_seeded.select(*target_cols)

display(df_target)


In [0]:
from pyspark.sql.functions import lower, regexp_replace

# Load Kaggle MTeamsSpellings.csv with alternate names
spellings_df = spark.read.option("header", True).csv("/Volumes/workspace/marchmadnesslandingzone/kaggle/MTeamSpellings.csv")

# Remove periods, hyphens and lower-case team name columns for join
spellings_df = spellings_df.withColumn(
    "TeamName_clean",
    lower(regexp_replace(regexp_replace(col("TeamNameSpelling"), r"[.]", ""), r"[-]", " "))
)

display(spellings_df)

# Join KenPom features with spelling alternatives
joined = kenpom_lower.join(spellings_df, kenpom_lower["Team_clean"] == spellings_df["TeamName_clean"], "left")

# Identify teams with missing Team_Id
no_id_teams = joined.filter(col("TeamID").isNull()).select("Team")\
    .drop('TeamNameSpelling', 'TeamName_clean')

# Note - manually added the following to the file in 2026.
# Cal St Bakersfield
# Mississippi Valley St
# Saint Francis
# Southeast Missouri St
# Texas A&M Corpus Chris

# Output results
print(f"Teams missing Team_Id: {no_id_teams.count()}")
display(no_id_teams)

display(joined)


In [0]:
joined_key_df = joined.withColumn('Season_Team', concat(col('Year'), F.lit('_'), col('TeamID')))

display(joined_key_df)


In [0]:
# Create schema if not exists
spark.sql("CREATE SCHEMA IF NOT EXISTS marchmadness")

# Save the transformed KenPom data to marchmadness.silver_kenpom
joined_key_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("marchmadness.silver_kenpom")

# Verify a sample of the saved table
display(spark.table("marchmadness.silver_kenpom"))
